In [1]:
pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
import re 

In [3]:
URL = "https://www.amazon.in/s?k=laptop&crid=3H4ZV8KKLMX73&sprefix=lapto%2Caps%2C348&ref=nb_sb_noss_2"

In [4]:
headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
"Chrome/144.0.0.0 Safari/537.36"}

In [5]:
data = []
print("Data list cleared")

for page in range(1, 7):  
    params = {"k": "laptop", "page": page}
    
    # Use verify=False to bypass SSL verification (for testing purposes)
    response = requests.get(URL, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    products = soup.find_all("div", {"data-component-type": "s-search-result"})

    
    
    for product in products:
        # Extract the product title
        title_tag = product.find("h2")
        if not title_tag:
            continue
        title_text = title_tag.get_text(strip=True)

        # Extract the product price
        price_tag = product.find("span", {"class": "a-price-whole"})
        price_text = price_tag.get_text(strip=True) if price_tag else "N/A"

        # Extract the Processor (Intel, AMD, Apple M, etc)
        processor = "N/A"
        
        # Try Intel processors - more flexible patterns
        intel_match = re.search(r'Intel\s+(?:Core\s+)?(?:i[3579]|m\d|Pentium|Celeron|Atom|Xeon)[-\w]*', title_text, re.IGNORECASE)
        if intel_match:
            processor = intel_match.group(0).strip()
        
        # Try AMD Ryzen/Athlon processors
        elif re.search(r'AMD', title_text, re.IGNORECASE):
            amd_match = re.search(r'AMD\s+(?:Ryzen|Athlon)[\s\w]*', title_text, re.IGNORECASE)
            if amd_match:
                processor = amd_match.group(0).strip()
        
        # Try Apple M series
        elif re.search(r'Apple\s+M', title_text, re.IGNORECASE):
            apple_match = re.search(r'Apple\s+M\d+\w*', title_text, re.IGNORECASE)
            if apple_match:
                processor = apple_match.group(0).strip()
        
        # Try other processors
        else:
            other_match = re.search(r'Qualcomm\s+Snapdragon|MediaTek|ARM|Exynos', title_text, re.IGNORECASE)
            if other_match:
                processor = other_match.group(0).strip()
        
        # Extract the product brand
        match = re.match(r'^\W*([A-Za-z]+)', title_text)
        brand = match.group(1).upper() if match else "UNKNOWN"

        # Extract RAM (improved version)
        ram_match = re.search(r'(\d+)\s*GB\s*(RAM|DDR\d+|LPDDR\d+)', title_text, re.IGNORECASE)
        ram = ram_match.group(1) + "GB" if ram_match else "N/A"
        
        # Extract the product rating 
        rating_tag = product.find("span", class_="a-icon-alt")
        rating = "N/A"
        if rating_tag:
            rating_match = re.search(r'(\d+\.?\d*)', rating_tag.get_text())
            rating = rating_match.group(1) if rating_match else "N/A"

        # Extract the Product SSD - Simplified
        ssd_match = re.search(r'(\d+)\s*(GB|TB)\s*(?:SSD|NVMe)', title_text, re.IGNORECASE) or re.search(r'(\d+)\s*(GB|TB)', title_text)
        ssd = (ssd_match.group(1) + ssd_match.group(2).upper()) if ssd_match else "N/A"
      

        # Extract the Product Windows Version - Improved version
        windows_version = "N/A"
        
        # Try: "Windows 11", "Windows 10", "Win11", "Win10", etc.
        windows_match = re.search(r'(?:Windows\s*|Win\s*)(\d+)', title_text, re.IGNORECASE)
        if windows_match:
            windows_version = "Windows " + windows_match.group(1)
        
        # Extract the Product Color
        color_match = re.search(r'\b(Black|Silver|Gray|Grey|White|Blue|Red|Gold|Green|Brown|Pink|Purple|Yellow|Orange|Champagne|Midnight|Space|Cosmic|Stardust|Graphite|Ash|Onyx|Platinum|Metallic)\b', title_text, re.IGNORECASE)
        color = color_match.group(1) if color_match else "N/A"

        # Extract Discount Percentage
        discount_tag = product.find("span", string=re.compile(r'%'))
        discount = discount_tag.get_text(strip=True) if discount_tag else "N/A"

        # Extract Screen Size
        screen_match = re.search(r'(\d{1,2}\.?\d?)\s*(?=[^\d]*cm|[^\d]*["\'])',title_text,re.IGNORECASE)
        screen_size = screen_match.group(1) if screen_match else "N/A"



        # Store the extracted data in a dictionary and append to the list
        data.append({
            "Title": title_text,
            "Price": price_text,
            "Processor":processor,
            "Brand": brand,
            'RAM': ram,
            "Rating": rating,
            "Storage": ssd,
            "Windows": windows_version,
            "Color": color,
            "Discount": discount,
            "Screen Size": screen_size
        })

      
    print(f"Page {page} scraped")
    time.sleep(1)

Data list cleared
Page 1 scraped
Page 2 scraped
Page 3 scraped
Page 4 scraped
Page 5 scraped
Page 6 scraped


In [10]:
for product in data:
    print("Title:", product["Title"])
    print("Price:", product["Price"])
    print("Brand:", product["Brand"])
    print("Processor:",product["Processor"])
    print("RAM:", product.get("RAM"))
    print("Rating:", product.get("Rating"))
    print("Storage:", product.get("Storage"))
    print("Windows:", product.get("Windows"))
    print("Color:", product.get("Color"))
    print("Discount:",product.get("Discount"))
    print("Screen Size:", product.get("Screen Size"))
    print("-" * 50)

Title: acer Aspire Lite, Intel Core 5 210H Processor, 16 GB RAM, 512 GB SSD, Full HD IPS, 15.6"/39.62 cm, Windows 11 Home, MSO, Pure Silver, 1.70 kgs, AL15-52H, Backlit Keyboard, Thin and Light Laptop
Price: 54,999
Brand: ACER
Processor: N/A
RAM: 16GB
Rating: 4.0
Storage: 512GB
Windows: Windows 11
Color: Silver
Discount: (31% off)
Screen Size: 15.6
--------------------------------------------------
Title: Dell 15, Intel Core i5 13th Gen-1334U, 16GB RAM, 1TB SSD, FHD, 15.6"/39.62cm, Windows 11, MSO 2024, Silver, 1.62kg, [Dell 15], Backlit Keyboard, 15 Month McAfee, Thin & Light, Laptop
Price: 60,526
Brand: DELL
Processor: Intel Core i5
RAM: 16GB
Rating: 3.7
Storage: 1TB
Windows: Windows 11
Color: Silver
Discount: (23% off)
Screen Size: 15.6
--------------------------------------------------
Title: EBook 11.6" HD Laptop | Best Student & Office Work Laptop | Celeron N4020 | 4GB DDR4 | 128GB eMMC + M.2 SSD Expandable Slot | Win 11 Home |31Wh Battery | UHD Graphics 600 | Black
Price: 10,990

In [11]:
df=pd.DataFrame(data)
df

,Title,Price,Processor,Brand,RAM,Rating,Storage,Windows,Color,Discount,Screen Size
0,"acer Aspire Lite, Intel Core 5 210H Processor,...","54,999",N/A,ACER,16GB,4.0,512GB,Windows 11,Silver,(31% off),15.6
1,"Dell 15, Intel Core i5 13th Gen-1334U, 16GB RA...","60,526",Intel Core i5,DELL,16GB,3.7,1TB,Windows 11,Silver,(23% off),15.6
2,"EBook 11.6"" HD Laptop | Best Student & Office ...","10,990",N/A,EBOOK,4GB,4.0,4GB,Windows 11,Black,(56% off),11.6
3,HP Chromebook X360 Intel Celeron N4020 14 inch...,"27,990",Intel Celeron,HP,4GB,3.9,4GB,N/A,N/A,(13% off),35.6
4,JioBook 11 with Lifetime Office | Android 4G L...,"10,990",Mediatek,JIOBOOK,4GB,2.9,4GB,N/A,Blue,(56% off),N/A
...,...,...,...,...,...,...,...,...,...,...,...
127,Acer TravelLite Thin Laptop AMD Ryzen 5 7430U ...,"36,999",AMD Ryzen 5 7430U,ACER,8GB,3.0,512GB,Windows 11,Black,(30% off),14
128,"ASUS Vivobook Go 15, AMD Ryzen 5 7520U Thin & ...","37,990",AMD Ryzen 5 7520U Thin,ASUS,8GB,4.0,512GB,Windows 11,Silver,(25% off),15.6
129,"Lenovo ThinkBook 16, AMD Ryzen 7 7735HS, 16GB ...","68,490",AMD Ryzen 7 7735HS,LENOVO,16GB,4.3,512GB,Windows 11,N/A,(26% off),16
130,"ASUS Vivobook S14,Smartchoice,AMD Ryzen AI 5 3...","65,990",AMD Ryzen AI 5 330,ASUS,16GB,4.4,512GB,Windows 11,Gray,(30% off),14


In [15]:
print(df.columns)

Index(['Product Name', 'Price', 'Processor', 'Brand', 'RAM', 'Rating',
       'Storage', 'Windows', 'Color', 'Discount', 'Screen Size'],
      dtype='object')


In [18]:
df.to_csv("amazon_laptops_raw_data.csv")